# Car Price Regression

This notebook follows the simple regression style from Machine Learning Zoomcamp: load data, normalize column names, split train/validation data, train a linear model, and measure RMSE on log price.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
df = pd.read_csv(Path('..') / 'data' / 'data.csv')
df.head()

In [ ]:
column_map = {
    'Make': 'make',
    'Model': 'model',
    'Year': 'year',
    'Engine Fuel Type': 'engine_fuel_type',
    'Engine HP': 'engine_hp',
    'Engine Cylinders': 'engine_cylinders',
    'Transmission Type': 'transmission_type',
    'Driven_Wheels': 'driven_wheels',
    'Number of Doors': 'number_of_doors',
    'Market Category': 'market_category',
    'Vehicle Size': 'vehicle_size',
    'Vehicle Style': 'vehicle_style',
    'highway MPG': 'highway_mpg',
    'city mpg': 'city_mpg',
    'Popularity': 'popularity',
    'MSRP': 'msrp',
}
df = df.rename(columns=column_map)
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.isnull().sum()

In [ ]:
features = [col for col in df.columns if col != 'msrp']
numeric_features = ['year', 'engine_hp', 'engine_cylinders', 'number_of_doors', 'highway_mpg', 'city_mpg', 'popularity']
categorical_features = [col for col in features if col not in numeric_features]

X = df[features]
y = np.log1p(df.msrp)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', min_frequency=10)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0)),
])
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
rmse